In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev = load_jsonl(absa_dev_path)
absa_test = load_jsonl(absa_test_path)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl'

In [ ]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}

In [ ]:
def build_conditioned_prompt(text: str, category: str):
    return (
        "أنت نظام تصنيف فقط، وليس مساعداً.\n"
        "مهمتك الوحيدة هي اختيار قيمة polarity صحيحة.\n\n"

        "سيتم إعطاؤك:\n"
        "- نص مراجعة فندق واحد\n"
        "- جانب واحد فقط (category)\n\n"

        "المطلوب:\n"
        "اختيار polarity هذا الجانب فقط.\n\n"

        "القيم المسموحة حصراً (اكتبها كما هي بالإنجليزية):\n"
        "positive | negative | neutral | conflict\n\n"

        "قواعد إلزامية صارمة:\n"
        "1) لا تكتب أي شرح أو تفسير أو تعليق.\n"
        "2) لا تذكر النص مرة أخرى.\n"
        "3) لا تذكر اسم الـ category.\n"
        "4) لا تستخدم كلمات عربية للمشاعر.\n"
        "5) لا تضف أي نص خارج JSON.\n"
        "6) الإخراج يجب أن يكون JSON فقط وعلى سطر واحد.\n"
        "7) يجب أن يبدأ الإخراج بـ { وينتهي بـ }.\n"
        "8) أي مخالفة لهذه القواعد تعتبر خطأ.\n\n"

        "صيغة الإخراج المطلوبة حرفياً:\n"
        f'{{"labels":[{{"category":"{category}","polarity":"positive"}}]}}\n\n'
        f"category: {category}\n"
        f"text: {text}\n"
        "output:\n"
    )

In [ ]:
def to_conditioned_sft_examples(r):
    exs = []
    for op in r["labels"]:
        cat = op["category"]
        pol = op["polarity"]
        prompt = build_conditioned_prompt(r["text"], cat)
        exs.append({
            "text": prompt + json.dumps({"labels":[{"category": cat, "polarity": pol}]}, ensure_ascii=False)
        })
    return exs

train_sft = []
for r in absa_train:
    train_sft.extend(to_conditioned_sft_examples(r))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch
model_path = "drive/MyDrive/FYP/ALLAM-7B"

def load_new_peft_model():
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.config.use_cache = False
    #Attach Lora
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=4,
        lora_alpha=8,
        lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    )
    model = get_peft_model(model, lora)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token_id is None:
      tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


In [ ]:
#model.print_trainable_parameters()


In [ ]:
# from datasets import Dataset
# from trl import SFTTrainer
# from transformers import TrainingArguments

# train_ds = Dataset.from_list(train_small_sft)
# dev_ds = Dataset.from_list(dev_sft)

# args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/FYP/slsa/allam_lora_slsa_HPT",
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     num_train_epochs=3,
#     logging_steps=50,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     fp16=True,
#     report_to="none",
# )

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_ds,
#     eval_dataset=dev_ds,
#     dataset_text_field="text",
#     max_seq_length=256,
#     args=args,
# )

# trainer.train()


In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 41.2 MB/s eta 0:00:00


In [ ]:
import os
OUT_BASE = "/content/drive/MyDrive/FYP/absa/allam_lora_conditioned_absa_peft"

In [ ]:
from datasets import Dataset
train_ds = Dataset.from_list(train_sft)

In [ ]:
import gc
from trl import SFTTrainer
from transformers import TrainingArguments

ModuleNotFoundError: No module named 'trl'

In [ ]:
def formatting_func(example):
    return example["text"]

In [ ]:
def train_one_run(lr, epochs):
  model, tokenizer = load_new_peft_model()

  args = TrainingArguments(
        output_dir=OUT_BASE,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=lr,
        num_train_epochs=epochs,
        logging_steps=50,
        save_strategy="no",
        fp16=True,
        report_to="none",
  )

  trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        formatting_func=formatting_func,
        args=args,
  )

  trainer.train()

  adapter_dir = os.path.join(OUT_BASE, "adapter")
  os.makedirs(adapter_dir, exist_ok=True)
  trainer.model.save_pretrained(adapter_dir)

  del trainer
  del model
  torch.cuda.empty_cache()
  gc.collect()

  return None


In [ ]:
absa_peft_eval = train_one_run(lr=5e-4, epochs=3)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Applying formatting function to train dataset:   0%|          | 0/7635 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/7635 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7635 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7635 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,1.080884
100,0.560548
150,0.510968
200,0.484756
250,0.438898
300,0.394080
350,0.354410
400,0.308492
450,0.280926
500,0.231713


100%|██████████| 452/452 [52:25<00:00,  6.96s/it]


#Fine tuned Model Eval -NOTEBOOK 2.2

In [ ]:
from peft import PeftModel

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
adapter_dir = "/content/drive/MyDrive/FYP/absa/allam_lora_conditioned_absa_peft/adapter"
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(64000, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [ ]:
def generate_chat(model, tokenizer, user_content: str):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,      # deterministic
            temperature=None,     # ignored when do_sample=False
            top_p=None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
import json
def extract_first_json(text):
    if not text:
        return None

    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                block = text[start:i+1]
                try:
                    return json.loads(block)
                except:
                    return None
    return None

##### Changed the Extract First Json because it couldnt track the {...} block because it was nested in here

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev = load_jsonl(absa_dev_path)
absa_test = load_jsonl(absa_test_path)

In [ ]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}
ABSA_POLARITY_SET = set(ABSA_POLARITIES)

In [ ]:
def build_conditioned_absa_prompt(text, category):
    return (
        "أنت نظام تصنيف فقط، وليس مساعداً.\n"
        "مهمتك الوحيدة هي اختيار قيمة polarity صحيحة.\n\n"

        "سيتم إعطاؤك:\n"
        "- نص مراجعة فندق واحد\n"
        "- جانب واحد فقط (category)\n\n"

        "المطلوب:\n"
        "اختيار polarity هذا الجانب فقط.\n\n"

        "القيم المسموحة حصراً (اكتبها كما هي بالإنجليزية):\n"
        "positive | negative | neutral | conflict\n\n"

        "قواعد إلزامية صارمة:\n"
        "1) لا تكتب أي شرح أو تفسير أو تعليق.\n"
        "2) لا تذكر النص مرة أخرى.\n"
        "3) لا تذكر اسم الـ category.\n"
        "4) لا تستخدم كلمات عربية للمشاعر.\n"
        "5) لا تضف أي نص خارج JSON.\n"
        "6) الإخراج يجب أن يكون JSON فقط وعلى سطر واحد.\n"
        "7) يجب أن يبدأ الإخراج بـ { وينتهي بـ }.\n"
        "8) أي مخالفة لهذه القواعد تعتبر خطأ.\n\n"

        "صيغة الإخراج المطلوبة حرفياً:\n"
        "{\"labels\":[{\"polarity\":\"positive\"}]}\n\n"

        f"category: {category}\n"
        f"text: {text}\n\n"
        "output:\n"
    )

In [ ]:
def parse_polarity_from_obj(obj):
    try:
        pol = obj["labels"][0]["polarity"]
        if isinstance(pol, str):
            pol = pol.strip().lower().replace(" ", "").replace("▁", "")
        return pol if pol in ABSA_POLARITY_SET else None
    except Exception:
        return None

def parse_polarity_from_text(raw):
    if not raw:
        return None
    s = raw.lower().replace(" ", "").replace("▁", "")
    for pol in ABSA_POLARITIES:
        if pol in s:
            return pol
    return None

def predict_conditioned_polarity(raw, category):
    obj = extract_first_json(raw)
    pol = parse_polarity_from_obj(obj) if obj else None
    if pol is None:
        pol = parse_polarity_from_text(raw)

    if pol is None:
        return None
    return (category, pol)

In [ ]:
from collections import defaultdict

def macro_f1_polarity(y_true, y_pred, labels=ABSA_POLARITIES):
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)

    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}

        for c, true_pol in true_map.items():
            pred_pol = pred_map.get(c)
            if pred_pol == true_pol:
                tp[true_pol] += 1
            else:
                fn[true_pol] += 1
                if pred_pol is not None:
                    fp[pred_pol] += 1

    f1s = []
    for pol in labels:
        p = tp[pol] / (tp[pol] + fp[pol]) if (tp[pol] + fp[pol]) else 0.0
        r = tp[pol] / (tp[pol] + fn[pol]) if (tp[pol] + fn[pol]) else 0.0
        f1 = (2*p*r)/(p+r) if (p+r) else 0.0
        f1s.append(f1)

    return sum(f1s) / len(f1s)


In [ ]:
def polarity_accuracy(y_true, y_pred):
    correct = total = 0
    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}
        for c, tp in true_map.items():
            total += 1
            if pred_map.get(c) == tp:
                correct += 1
    return correct / total if total else 0.0

In [ ]:
from tqdm import tqdm

In [ ]:
def eval_absa_conditioned(rows):
    bad = 0
    y_pred, y_true = [], []

    for r in tqdm(rows):
        gold_pairs = [(x["category"], x["polarity"]) for x in r["labels"]]
        gold_cats = sorted({cat for cat, _ in gold_pairs})

        pred_set = set()

        for cat in gold_cats:
            prompt = build_conditioned_absa_prompt(r["text"], cat)

            raw = generate_chat(model, tokenizer, prompt)

            print("\nRAW:", raw)

            pred_item = predict_conditioned_polarity(raw, cat)
            print(pred_item)
            if pred_item is None:
                bad += 1
                continue

            pred_set.add(pred_item)

        y_true.append(set(gold_pairs))
        y_pred.append(pred_set)

    return {
        "macro-f1": macro_f1_polarity(y_true, y_pred),
        "accuracy": polarity_accuracy(y_true, y_pred),
        "invalid": bad,
        "n": len(rows),
    }

In [ ]:
zeroS = eval_absa_conditioned(absa_test)

  0%|          | 0/452 [00:00<?, ?it/s]


RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " F AC IL IT IES # PR IC ES "} ]
('FACILITIES#PRICES', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " H OT EL # GEN ER AL "} ]}
('HOTEL#GENERAL', 'negative')


  0%|          | 1/452 [00:05<38:06,  5.07s/it]


RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " L OC AT ION # GEN ER AL "} ]}
('LOCATION#GENERAL', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " pos it ive ", " ca te go ry ": " H OT EL # GEN ER AL "} ]}
('HOTEL#GENERAL', 'positive')


  0%|          | 2/452 [00:07<27:42,  3.69s/it]

...


RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " S ER VI CE # GEN ER AL "} ]}
('SERVICE#GENERAL', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " F AC IL IT IES # GEN ER AL "} ]}
('FACILITIES#GENERAL', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " F AC IL IT IES # MI SC ELL AN EO US
('FACILITIES#MISCELLANEOUS', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " H OT EL # GEN ER AL "} ]}
('HOTEL#GENERAL', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " L OC AT ION # GEN ER AL "} ]}
('LOCATION#GENERAL', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " R OO MS # GEN ER AL "} ]}
('ROOMS#GENERAL', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " R OO MS _ AM EN IT IES # DE SI GN
('ROOMS_AMENITIES#DESIGN_FEATURES', 'negative')

RAW: {" la 

100%|█████████▉| 451/452 [49:20<00:09,  9.05s/it]


RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " S ER VI CE # GEN ER AL "} ]}
('SERVICE#GENERAL', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " H OT EL # CL EA NL IN ESS "} ]}
('HOTEL#CLEANLINESS', 'negative')

RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " H OT EL # GEN ER AL "} ]}
('HOTEL#GENERAL', 'negative')


100%|██████████| 452/452 [49:24<00:00,  6.56s/it]


RAW: {" la be ls ": [ {" pol ar ity ": " neg at ive ", " ca te go ry ": " S ER VI CE # GEN ER AL "} ]}
('SERVICE#GENERAL', 'negative')


In [ ]:
print(zeroS)

{'macro-f1': 0.4547855273776411, 'accuracy': 0.8257645968489342, 'invalid': 0, 'n': 452}


In [ ]:
results = {
    "zeroS": zeroS
}

In [ ]:
results_path = "drive/MyDrive/FYP/absa/allam_lora_conditioned_absa_peft/results/results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False)

In [ ]:
from google.colab import runtime
runtime.unassign()

#Analysis
The PEFT model significantly outperformed the base prompting approach. Accuracy increased from 72.15% to 82.58%, while macro-F1 improved from 0.38 to 0.45. The increase in macro-F1 indicates improved performance across multiple aspect-polarity classes rather than gains limited to dominant categories. These findings suggest that parameter-efficient fine-tuning allows the model to better internalize the aspect schema and sentiment distinctions required for structured ABSA. However, the remaining gap between accuracy and macro-F1 indicates persistent challenges related to class imbalance and fine-grained category differentiation.

# Compared to Zero Shot
The base zero-shot model performs competitively for aspect-conditioned ABSA, while RAG provides minimal benefit and can degrade performance due to retrieval noise and leakage.